In [6]:
import os
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW

In [7]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")

Устройство: cuda


In [8]:
# Гиперпараметры

MODEL_NAME = "distilbert-base-uncased"

MAX_LENGTH = 64
# По данным EDA: средний запрос ~8 слов, максимум ~28 слов
# 64 токена полностью покрывает все запросы датасета

BATCH_SIZE = 32  # Количество примеров обрабатываемых за один шаг обучения.
# Малый батч (16) даёт зашумлённый градиент и нестабильное обучение.
# Большой батч (64) может превысить объём памяти GPU.
# 32 — стандартный компромисс для задач классификации текста.

LEARNING_RATE = 2e-5  # Размер шага обновления весов на каждой итерации.
# Значение 2e-5 рекомендовано авторами BERT для задач fine-tuning
# и минимизирует риск потери знаний полученных на этапе предобучения.

WEIGHT_DECAY = 0.01  # Коэффициент L2-регуляризации для предотвращения переобучения.
# Стандартное значение для задач fine-tuning

NUM_EPOCHS = 5  # Максимальное количество полных проходов по обучающей выборке
# При fine-tuning трансформеров более 5 эпох как правило приводит к переобучению

WARMUP_RATIO = 0.1  # Доля шагов обучения отведённых под линейный разогрев learning rate.
# В течение первых 10% шагов lr плавно растёт от 0 до максимума, затем линейно убывает.

EARLY_STOPPING_PATIENCE = 2  # Количество эпох без улучшения val_loss после которых обучение останавливается.
# Предотвращает переобучение и экономит вычислительные ресурсы.

DATA_DIR = Path("../../data/splits")
RESULTS_DIR = Path("../../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [9]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
num_classes = train_df["intent"].nunique()

print(f"Train: {len(train_df)} примеров")
print(f"Val:   {len(val_df)} примеров")
print(f"Классов: {num_classes}")
train_df.head(3)

Train: 16694 примеров
Val:   3578 примеров
Классов: 151


,text,intent
0,don't do this process,147
1,how do i use the reward for my bank of hawaii,6
2,what time is today's meeting,19


In [10]:
class CLINCDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts  = df["text"].tolist() # список всех запросов
        self.labels = df["intent"].tolist() # список всех меток
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [11]:
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

train_dataset = CLINCDataset(train_df, tokenizer)
val_dataset   = CLINCDataset(val_df, tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,   # перемешиваем train в каждой эпохе
                    # это важно потому что если модель всегда видит примеры
                    # в одном порядке — она может запомнить порядок, а не смысл.
    num_workers=0
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,  # val не перемешиваем — порядок не важен
    num_workers=0
)

print(f"Шагов в эпохе: {len(train_loader)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Шагов в эпохе: 522


In [12]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes
)
model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Всего параметров:    {total_params:,}")
print(f"Обучаемых параметров: {trainable_params:,}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Всего параметров:    67,069,591
Обучаемых параметров: 67,069,591


In [13]:
no_decay = ["bias", "LayerNorm.weight"]  # bias и LayerNorm не регуляризуем

optimizer_grouped_parameters = [
    {
        "params": [p for n, p in model.named_parameters()
                   if not any(nd in n for nd in no_decay)],
        "weight_decay": WEIGHT_DECAY,
    },
    {
        "params": [p for n, p in model.named_parameters()
                   if any(nd in n for nd in no_decay)],
        "weight_decay": 0.0,
    },
]

optimizer = AdamW(optimizer_grouped_parameters, lr=LEARNING_RATE)

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"Всего шагов:   {total_steps}")
print(f"Шагов разогрева: {warmup_steps}")

Всего шагов:   2610
Шагов разогрева: 261


In [14]:
import time

best_val_loss = float("inf")
patience_counter = 0
training_history = []

for epoch in range(1, NUM_EPOCHS + 1):
    # Обучение
    model.train()
    train_loss = 0.0
    epoch_start = time.time()

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Валидация
    model.eval()
    val_loss = 0.0
    correct  = 0
    total    = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids = input_ids,
                attention_mask = attention_mask,
                labels = labels
            )

            val_loss += outputs.loss.item()
            preds = outputs.logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total
    epoch_time = time.time() - epoch_start

    training_history.append({
        "epoch": epoch,
        "train_loss": round(train_loss, 4),
        "val_loss": round(val_loss, 4),
        "val_acc": round(val_acc, 4),
    })

    print(f"Эпоха {epoch}/{NUM_EPOCHS} | "
          f"train_loss={train_loss:.4f} | "
          f"val_loss={val_loss:.4f} | "
          f"val_acc={val_acc:.4f} | "
          f"время={epoch_time:.0f}с")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        model.save_pretrained(RESULTS_DIR / "models/finetuned_distilbert")
        tokenizer.save_pretrained(RESULTS_DIR / "models/finetuned_distilbert")
        print(f"Модель сохранена (val_loss={val_loss:.4f})")
    else:
        patience_counter += 1
        print(f"Нет улучшения ({patience_counter}/{EARLY_STOPPING_PATIENCE})")
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping на эпохе {epoch}")
            break

print("Обучение завершено")

Эпоха 1/5 | train_loss=4.3211 | val_loss=2.9397 | val_acc=0.5699 | время=102с


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель сохранена (val_loss=2.9397)
Эпоха 2/5 | train_loss=2.1092 | val_loss=1.2153 | val_acc=0.8700 | время=105с


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель сохранена (val_loss=1.2153)
Эпоха 3/5 | train_loss=0.9426 | val_loss=0.6120 | val_acc=0.9290 | время=105с


...

Модель сохранена (val_loss=0.6120)
Эпоха 4/5 | train_loss=0.5117 | val_loss=0.4183 | val_acc=0.9391 | время=105с


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель сохранена (val_loss=0.4183)
Эпоха 5/5 | train_loss=0.3604 | val_loss=0.3704 | val_acc=0.9413 | время=105с


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель сохранена (val_loss=0.3704)
Обучение завершено


In [18]:
# Сохраняем лог обучения для будующей оценки
import json

with open(RESULTS_DIR / "exp01_training_log.json", "w") as f:
    json.dump(training_history, f, indent=2)